# LogisticRegression 逻辑回归模型演示

本 Notebook 演示 `LogisticRegression` 类的完整功能，包括：
- 基础模型训练与预测
- 统计信息计算（标准误差、p值、VIF）
- 回归结果摘要输出
- 显著性检验与多重共线性检测

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from hscredit.core.models import LogisticRegression

## 1. 创建示例数据

In [2]:
# 创建示例数据
np.random.seed(42)
n_samples = 2000

data = pd.DataFrame({
    'age': np.random.randint(18, 65, n_samples),
    'income': np.random.randint(3000, 50000, n_samples),
    'score': np.random.randint(300, 800, n_samples),
    'debt_ratio': np.random.uniform(0.1, 0.8, n_samples),
})

# 创建目标变量（基于逻辑关系）
logits = (
    -5 + 
    0.05 * data['age'] + 
    0.0001 * data['income'] + 
    -0.01 * data['score'] + 
    2 * data['debt_ratio'] +
    np.random.randn(n_samples) * 0.5
)
prob = 1 / (1 + np.exp(-logits))
data['target'] = (prob > 0.5).astype(int)

print(f"数据集样本数: {len(data)}")
print(f"特征数量: {len(data.columns) - 1}")
print(f"坏样本率: {data['target'].mean():.2%}")
data.head()

数据集样本数: 2000
特征数量: 4
坏样本率: 0.65%


,age,income,score,debt_ratio,target
0,56,38984,421,0.561102,0
1,46,29616,576,0.472604,0
2,32,5113,345,0.172286,0
3,60,46006,752,0.220767,0
4,25,16760,689,0.497582,0


In [3]:
# 划分训练集和测试集
X = data.drop('target', axis=1)
y = data['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"训练集样本数: {len(X_train)}")
print(f"测试集样本数: {len(X_test)}")

训练集样本数: 1400
测试集样本数: 600


## 2. 基础模型训练

In [4]:
# 创建并训练模型
model = LogisticRegression(
    calculate_stats=True,  # 启用统计信息计算
    C=1.0,                 # 正则化强度
    max_iter=1000,         # 最大迭代次数
    random_state=42
)

model.fit(X_train, y_train)
print("模型训练完成!")

模型训练完成!


In [5]:
# 模型预测
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# 评估模型
auc = roc_auc_score(y_test, y_proba)
print(f"测试集 AUC: {auc:.4f}")
print("\n分类报告:")
print(classification_report(y_test, y_pred, target_names=['好样本', '坏样本']))

测试集 AUC: 0.9954

分类报告:
              precision    recall  f1-score   support

         好样本       1.00      1.00      1.00       596
         坏样本       0.50      0.50      0.50         4

    accuracy                           0.99       600
   macro avg       0.75      0.75      0.75       600
weighted avg       0.99      0.99      0.99       600



## 3. 统计摘要输出

In [14]:
# 获取回归统计摘要
summary = model.summary()
print("回归统计摘要:")
summary

回归统计摘要:


,Coef.,Std.Err,z,P>|z|,[0.025,0.975],VIF
const,-0.088035,7.794471,-0.011295,0.990988,-15.365199,15.189128,33.624707
age,0.222514,0.082512,2.696744,0.007002,0.060790,0.384238,1.000981
income,0.000496,0.000177,2.806294,0.005011,0.000150,0.000843,1.000758
score,-0.101052,0.040189,-2.514409,0.011923,-0.179823,-0.022281,1.000536
debt_ratio,1.259687,2.185730,0.576323,0.564397,-3.024343,5.543718,1.000247


### 3.1 带特征描述的摘要

In [17]:
# 添加特征描述
feature_map = {
    'age': '客户年龄',
    'income': '月收入',
    'score': '信用评分',
    'debt_ratio': '债务收入比'
}

summary_with_desc = model.summary_with_desc(feature_map)
summary_with_desc #[['Features', 'Describe', 'Coef.', 'P>|z|', 'VIF']]

,Features,Describe,Coef.,Std.Err,z,P>|z|,[0.025,0.975],VIF
0,const,,-0.088035,7.794471,-0.011295,0.990988,-15.365199,15.189128,33.624707
1,age,客户年龄,0.222514,0.082512,2.696744,0.007002,0.060790,0.384238,1.000981
2,income,月收入,0.000496,0.000177,2.806294,0.005011,0.000150,0.000843,1.000758
3,score,信用评分,-0.101052,0.040189,-2.514409,0.011923,-0.179823,-0.022281,1.000536
4,debt_ratio,债务收入比,1.259687,2.185730,0.576323,0.564397,-3.024343,5.543718,1.000247


## 4. 显著性检验

In [18]:
# 获取显著特征（p < 0.05）
sig_features = model.get_significant_features(alpha=0.05)
print("显著特征 (p < 0.05):")
sig_features[['Coef.', 'Std.Err', 'P>|z|']]

显著特征 (p < 0.05):


,Coef.,Std.Err,P>|z|
age,0.222514,0.082512,0.007002
income,0.000496,0.000177,0.005011
score,-0.101052,0.040189,0.011923


In [19]:
# 获取高度显著特征（p < 0.01）
sig_features_01 = model.get_significant_features(alpha=0.01)
print("高度显著特征 (p < 0.01):")
print(sig_features_01[['Coef.', 'P>|z|']])

高度显著特征 (p < 0.01):
           Coef.     P>|z|
age     0.222514  0.007002
income  0.000496  0.005011


## 5. 多重共线性检测

In [20]:
# 检查所有特征的VIF
print("所有特征的VIF值:")
vif_table = model.summary()[['VIF']].copy()
vif_table = vif_table[vif_table.index != 'const']
print(vif_table)

print(f"\n平均VIF: {vif_table['VIF'].mean():.4f}")
print(f"最大VIF: {vif_table['VIF'].max():.4f}")

所有特征的VIF值:
                 VIF
age         1.000981
income      1.000758
score       1.000536
debt_ratio  1.000247

平均VIF: 1.0006
最大VIF: 1.0010


In [21]:
# 检查高共线性特征（VIF > 5）
high_vif = model.check_multicollinearity(threshold=5.0)
if len(high_vif) > 0:
    print("高共线性特征 (VIF > 5):")
    print(high_vif)
else:
    print("未发现高共线性特征 (所有 VIF <= 5)")

未发现高共线性特征 (所有 VIF <= 5)


## 6. 不同正则化强度对比

In [22]:
# 对比不同 C 值（正则化强度）的效果
C_values = [0.01, 0.1, 1.0, 10.0]
results = []

for C in C_values:
    model_c = LogisticRegression(C=C, calculate_stats=True, max_iter=1000, random_state=42)
    model_c.fit(X_train, y_train)
    
    y_proba_c = model_c.predict_proba(X_test)[:, 1]
    auc_c = roc_auc_score(y_test, y_proba_c)
    
    summary_c = model_c.summary()
    max_vif = summary_c['VIF'][summary_c.index != 'const'].max()
    
    results.append({
        'C': C,
        'AUC': auc_c,
        'Max VIF': max_vif
    })

results_df = pd.DataFrame(results)
print("不同正则化强度的效果对比:")
print(results_df)

不同正则化强度的效果对比:
       C       AUC   Max VIF
0   0.01  0.994547  1.000981
1   0.10  0.994966  1.000981
2   1.00  0.995386  1.000981
3  10.00  0.998322  1.000981


## 7. 样本权重使用

In [24]:
# 使用样本权重（增加少数类的权重）
class_ratio = y_train.value_counts()[0] / y_train.value_counts()[1]
sample_weight = np.where(y_train == 1, class_ratio, 1.0)

model_weighted = LogisticRegression(
    calculate_stats=True,
    max_iter=1000,
    random_state=42
)
model_weighted.fit(X_train, y_train, sample_weight=sample_weight)

y_proba_weighted = model_weighted.predict_proba(X_test)[:, 1]
auc_weighted = roc_auc_score(y_test, y_proba_weighted)

print(f"带样本权重的 AUC: {auc_weighted:.4f}")
print(f"不带样本权重的 AUC: {auc:.4f}")

print("\n带权重的回归系数:")
summary_weighted = model_weighted.summary()
print(summary_weighted[['Coef.', 'P>|z|']])

带样本权重的 AUC: 0.9975
不带样本权重的 AUC: 0.9954

带权重的回归系数:
                Coef.     P>|z|
const      -12.184278  0.225643
age          0.433491  0.002583
income       0.001102  0.001170
score       -0.166018  0.004580
debt_ratio   3.878340  0.265537


## 8. 总结

**LogisticRegression 主要功能**：
- `fit(X, y)`: 训练模型，自动计算统计信息
- `predict(X)`: 预测类别标签
- `predict_proba(X)`: 预测概率
- `summary()`: 输出回归统计摘要
- `summary_with_desc(feature_map)`: 带特征描述的摘要
- `get_significant_features(alpha)`: 获取显著特征
- `check_multicollinearity(threshold)`: 检测多重共线性

**统计指标说明**：
- `Coef.`: 回归系数
- `Std.Err`: 标准误差
- `z`: z统计量
- `P>|z|`: p值
- `[0.025 / 0.975]`: 95%置信区间
- `VIF`: 方差膨胀因子（检测共线性）